# Phase 4 — raw-file KD with fixed loss (depth12/experts80)

Isolates **training-data shape**: same fixed KD loss (PR #71) and hyperparameters,
but the **original** `bonsai-teacher-logits-baseline` raw-file cache (not function-mode).
If function extraction were necessary, this arm should underperform function-mode fixed-loss;
if the loss fix was the whole story, expect a similar **best-val** pass@1.

In [1]:
import json
from pathlib import Path

DEBUG = Path("../docs/debug")
PREFIX = "phase4_depth12_experts80_rawfile_fixedloss"
summary = json.loads((DEBUG / f"{PREFIX}_run_summary.json").read_text())
summary

{'kd_wall_clock_sec': 8405, 'kd_wall_clock_human': '2h 20m 5s', 'val_best_step': 239, 'val_best': 0.27849485784769057, 'val_final_step': 2999, 'val_final': 0.5260525114834309, 'humaneval_final_pass_at_1': 0.35, 'humaneval_best_pass_at_1': 0.55, 'distill_run': 'runs/20261011T182939Z-cba49b-distill-depth12-experts80-rawfile-vs-baseline-3000-warmup450-lr1e4-fixedloss', 'humaneval_final_run': 'runs/20261011T204930Z-1550a0-distill-depth12-experts80-rawfile-fixedloss-humaneval-ts-n40-final', 'humaneval_best_run': 'runs/20261011T210006Z-cdd44b-distill-depth12-experts80-rawfile-fixedloss-humaneval-ts-n40-best'}

In [2]:
import matplotlib.pyplot as plt

vals = []
for line in (DEBUG / f"{PREFIX}_kd_metrics.jsonl").read_text().splitlines():
    row = json.loads(line)
    if "val_loss" in row:
        vals.append((row["step"], row["val_loss"]))
steps, losses = zip(*vals)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, losses, linewidth=1)
ax.axvline(summary["val_best_step"], color="C1", linestyle="--", label=f"best @ {summary['val_best_step']}")
ax.set_xlabel("step")
ax.set_ylabel("val KD loss")
ax.set_title("Fixed-loss raw-file KD")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<Figure size 800x400 with 1 Axes>

In [2]:
comparison = {
    "pruned_only": 0.475,
    "raw_file_broken_loss_kd": 0.0,
    "function_mode_fixed_loss_best": 0.55,
    "raw_file_fixed_loss_final": summary["humaneval_final_pass_at_1"],
    "raw_file_fixed_loss_best": summary["humaneval_best_pass_at_1"],
}
comparison

{'pruned_only': 0.475, 'raw_file_broken_loss_kd': 0.0, 'function_mode_fixed_loss_best': 0.55, 'raw_file_fixed_loss_final': 0.35, 'raw_file_fixed_loss_best': 0.55}

In [3]:
gens = json.loads(
    (DEBUG / f"{PREFIX}_humaneval_generations_best_multiple-ts.json").read_text()
)
for i in range(3):
    text = gens[i][0]
    print("=" * 72)
    print(f"best-val problem {i} (len={len(text)})")
    print(text[:1200])
    if len(text) > 1200:
        print("... [truncated]")

best-val problem 0 (len=555)
//Check if in given array of numbers, are any two numbers closer to each other than
// given threshold.
// >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
// false
// >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
// true
function has_close_elements(numbers: number[], threshold: number): boolean {
    for (let i = 0; i < numbers.length; i++) {
        for (let j = i + 1; j < numbers.length; j++) {
            if (Math.abs(numbers[i] - numbers[j]) < threshold) {
                return true;
            }
        }
    }
    return false;
}

best-val problem 1 (len=988)
//Input to this function is a string containing multiple groups of nested parentheses. Your goal is to
// separate those group into separate strings and return the array of those.
// Separate groups are balanced (each open brace is properly closed) and not nested within each other
// Ignore any spaces in the input string.
// >>> separate_paren_groups("( ) (( )) (( )( ))")
// ["()", "